In [ ]:
!pip install -q transformers

In [ ]:

import json
from pathlib import Path
from collections import Counter

path = Path("/content/paired_aes_manifest_reconstructed.json")
assert path.exists(), "Upload the manifest to Colab first"

with path.open(encoding="utf-8-sig") as f:
    records = json.load(f)

print("Records:", len(records))
print("Labels:", dict(Counter(r["label"] for r in records)))
print("Benchmarks:", len(set(r["benchmark"] for r in records)))
print("AES-T600 present:", any(
    r["benchmark"] == "AES-T600" for r in records
))
print("Missing graph paths:", sum(
    not r.get("merged_path") for r in records
))


Records: 52
Labels: {0: 26, 1: 26}
Benchmarks: 26
AES-T600 present: False
Missing graph paths: 0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/codebert-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded:", MODEL_NAME)
print("Hidden size:", model.config.hidden_size)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Device: cuda
GPU: Tesla T4
Model loaded: microsoft/codebert-base
Hidden size: 768


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving AES_RTL_sources.zip to AES_RTL_sources.zip


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/AES_RTL_sources.zip")

if not zip_path.exists():
    print("ZIP not found. Uploaded files:", list(Path("/content").glob("*.zip")))
else:
    with zipfile.ZipFile(zip_path, "r") as z:
        bad_file = z.testzip()
        names = z.namelist()

    print("ZIP integrity:", "PASS" if bad_file is None else f"FAIL: {bad_file}")
    print("Total archived files:", len(names))
    print("Sample paths:")
    print(*names[:12], sep="\n")

    clean = {
        p.split("/")[4]
        for p in names
        if "/clean/" in p and p.lower().endswith((".v", ".sv", ".vh", ".svh"))
    }
    trojan = {
        p.split("/")[4]
        for p in names
        if "/trojan/" in p and p.lower().endswith((".v", ".sv", ".vh", ".svh"))
    }

    print("Clean benchmark folders detected:", len(clean))
    print("Trojan benchmark folders detected:", len(trojan))

ZIP integrity: PASS
Total archived files: 395
Sample paths:
AES\clean\
AES\trojan\
AES\clean\AES-T100\aes_128.v
AES\clean\AES-T100\Read_Me.txt
AES\clean\AES-T100\round.v
AES\clean\AES-T100\simulation.do
AES\clean\AES-T100\table.v
AES\clean\AES-T100\test_aes_128.v
AES\clean\AES-T1000\aes_128.v
AES\clean\AES-T1000\Read_Me.txt
AES\clean\AES-T1000\round.v
AES\clean\AES-T1000\simulation.do
Clean benchmark folders detected: 0
Trojan benchmark folders detected: 0


In [ ]:
# ============================================================
# AES HARDWARE TROJAN DETECTION
# CodeBERT RTL Embedding Generation — Version 2
# ============================================================

import json
import hashlib
import zipfile
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# -------------------- CONFIGURATION --------------------------

SOURCE_ROOT = Path("/content/rtl_source_data/AES")
OUTPUT_ROOT = Path("/content/rtl_embeddings_v2")
ZIP_PATH = Path("/content/rtl_embeddings_v2.zip")

MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
STRIDE = 50
BATCH_SIZE = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXTENSIONS = {".v", ".sv", ".vh", ".svh"}

# -------------------- SAFETY CHECKS --------------------------

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_ROOT}")

for class_name in ("clean", "trojan"):
    class_root = SOURCE_ROOT / class_name
    if not class_root.is_dir():
        raise FileNotFoundError(f"Missing directory: {class_root}")

# Never overwrite an earlier run.
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise FileExistsError(
        f"{OUTPUT_ROOT} already contains files. "
        "Use a new output directory for another run; "
        "do not delete the old results blindly."
    )

if ZIP_PATH.exists():
    raise FileExistsError(
        f"{ZIP_PATH} already exists. Rename the previous ZIP "
        "or change ZIP_PATH before starting."
    )

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------- LOAD CODEBERT --------------------------

print("\nLoading CodeBERT...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

assert model.config.hidden_size == 768
print("Model:", MODEL_NAME)
print("Hidden size:", model.config.hidden_size)

# -------------------- HELPERS --------------------------------

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def is_testbench(path: Path) -> bool:
    """Conservative filename-based testbench exclusion."""
    name = path.name.lower()
    return (
        name.startswith("test")
        or name.startswith("tb")
        or "_tb." in name
        or "testbench" in name
    )


def discover_designs():
    """Discover source files and assemble one deterministic text per design."""
    designs = []

    for class_name, label in (("clean", 0), ("trojan", 1)):
        class_root = SOURCE_ROOT / class_name
        benchmarks = sorted(
            p for p in class_root.iterdir() if p.is_dir()
        )

        if len(benchmarks) != 27:
            raise RuntimeError(
                f"Expected 27 {class_name} benchmarks; found {len(benchmarks)}"
            )

        for bench in benchmarks:
            all_files = sorted(
                p for p in bench.rglob("*")
                if p.is_file() and p.suffix.lower() in EXTENSIONS
            )

            included = [p for p in all_files if not is_testbench(p)]
            excluded = [p for p in all_files if is_testbench(p)]

            if not included:
                raise RuntimeError(
                    f"No RTL files selected for {class_name}/{bench.name}"
                )

            # File names, benchmark IDs, and labels are NOT inserted into text.
            source_text = "\n\n".join(
                p.read_text(encoding="utf-8", errors="replace")
                for p in included
            )

            designs.append({
                "class_name": class_name,
                "label": label,
                "benchmark": bench.name,
                "included_files": included,
                "excluded_files": excluded,
                "source_text": source_text,
                "source_hash": sha256_bytes(source_text.encode("utf-8")),
            })

    return designs


def embed_source(source_text: str) -> torch.Tensor:
    """Return shape (num_chunks, 768) using masked mean pooling."""
    encoded = tokenizer(
        source_text,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding="max_length",
        return_attention_mask=True,
    )

    input_ids = encoded["input_ids"]
    attention_masks = encoded["attention_mask"]

    # Normalize a single-chunk result to a list of chunks.
    if input_ids and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_masks = [attention_masks]

    pooled_chunks = []

    with torch.inference_mode():
        for start in range(0, len(input_ids), BATCH_SIZE):
            ids = torch.tensor(
                input_ids[start:start + BATCH_SIZE],
                dtype=torch.long,
                device=DEVICE,
            )

            mask = torch.tensor(
                attention_masks[start:start + BATCH_SIZE],
                dtype=torch.long,
                device=DEVICE,
            )

            hidden = model(
                input_ids=ids,
                attention_mask=mask,
            ).last_hidden_state

            mask_float = mask.unsqueeze(-1).to(hidden.dtype)

            pooled = (
                (hidden * mask_float).sum(dim=1)
                / mask_float.sum(dim=1).clamp(min=1.0)
            )

            pooled_chunks.append(pooled.cpu().float())

    if not pooled_chunks:
        raise RuntimeError("No embedding chunks generated.")

    return torch.cat(pooled_chunks, dim=0).contiguous()


# -------------------- DISCOVER DATA --------------------------

designs = discover_designs()

print("\nDataset verified")
print("Total designs:", len(designs))
print("Clean designs:", sum(d["label"] == 0 for d in designs))
print("Trojan designs:", sum(d["label"] == 1 for d in designs))
print(
    "Distinct combined source texts:",
    len({d["source_hash"] for d in designs}),
)

# -------------------- GENERATE EMBEDDINGS --------------------

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

# Identical source texts reuse the same computed embedding.
embedding_cache = {}
manifest_rows = []

for index, d in enumerate(
    tqdm(designs, desc="Generating CodeBERT embeddings"),
    start=1,
):
    source_hash = d["source_hash"]

    if source_hash not in embedding_cache:
        embedding_cache[source_hash] = embed_source(d["source_text"])

    tensor = embedding_cache[source_hash]

    out_dir = OUTPUT_ROOT / d["class_name"]
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{d['benchmark']}.pt"
    torch.save(tensor, out_path)

    included_file_hashes = {
        p.relative_to(SOURCE_ROOT).as_posix():
            sha256_bytes(p.read_bytes())
        for p in d["included_files"]
    }

    row = {
        "class_name": d["class_name"],
        "label": d["label"],
        "benchmark": d["benchmark"],
        "source_text_sha256": source_hash,
        "included_files": [
            p.relative_to(SOURCE_ROOT).as_posix()
            for p in d["included_files"]
        ],
        "included_file_sha256": included_file_hashes,
        "excluded_testbench_files": [
            p.relative_to(SOURCE_ROOT).as_posix()
            for p in d["excluded_files"]
        ],
        "num_chunks": int(tensor.shape[0]),
        "embedding_dim": int(tensor.shape[1]),
        "embedding_path": out_path.relative_to(OUTPUT_ROOT).as_posix(),
    }

    manifest_rows.append(row)

    print(
        f"{index:02d}/54 | {d['class_name']:6s} | "
        f"{d['benchmark']:10s} | "
        f"chunks={tensor.shape[0]} | "
        f"{'cached' if sum(r['source_text_sha256'] == source_hash for r in manifest_rows[:-1]) else 'computed/reused'}"
    )

# -------------------- SAVE MANIFEST --------------------------

manifest = {
    "model": MODEL_NAME,
    "device": str(DEVICE),
    "max_length": MAX_LENGTH,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "pooling": (
        "Attention-mask-weighted mean of final hidden states "
        "for each tokenizer chunk"
    ),
    "source_root": str(SOURCE_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "filename_and_label_inserted_into_model_input": False,
    "testbench_exclusion_rule": (
        "Filename starts with test or tb, contains _tb., "
        "or contains testbench"
    ),
    "unique_source_texts": len(embedding_cache),
    "designs": manifest_rows,
}

manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

# -------------------- VALIDATE OUTPUTS ------------------------

assert len(manifest_rows) == 54
assert sum(r["label"] == 0 for r in manifest_rows) == 27
assert sum(r["label"] == 1 for r in manifest_rows) == 27

for row in manifest_rows:
    tensor_path = OUTPUT_ROOT / row["embedding_path"]

    tensor = torch.load(
        tensor_path,
        map_location="cpu",
        weights_only=True,
    )

    assert tensor.ndim == 2
    assert tensor.shape[1] == 768
    assert tensor.shape[0] == row["num_chunks"]
    assert torch.isfinite(tensor).all()

print("\nAll 54 embedding tensors passed validation.")

# -------------------- PACKAGE FOR DOWNLOAD -------------------

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_ROOT.parent),
            )

print("\n========== COMPLETE ==========")
print("Designs:", len(manifest_rows))
print("Clean:", sum(r["label"] == 0 for r in manifest_rows))
print("Trojan:", sum(r["label"] == 1 for r in manifest_rows))
print("Unique source texts:", len(embedding_cache))
print("Embedding dimension: 768")
print("Output directory:", OUTPUT_ROOT)
print("Manifest:", manifest_path)
print("Download package:", ZIP_PATH)

Device: cuda
GPU: Tesla T4

Loading CodeBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model: microsoft/codebert-base
Hidden size: 768

Dataset verified
Total designs: 54
Clean designs: 27
Trojan designs: 27
Distinct combined source texts: 29


Generating CodeBERT embeddings:   0%|          | 0/54 [00:00<?, ?it/s]

01/54 | clean  | AES-T100   | chunks=2 | computed/reused
02/54 | clean  | AES-T1000  | chunks=2 | cached
03/54 | clean  | AES-T1100  | chunks=2 | cached
04/54 | clean  | AES-T1200  | chunks=2 | cached
05/54 | clean  | AES-T1300  | chunks=2 | cached
06/54 | clean  | AES-T1400  | chunks=2 | cached
07/54 | clean  | AES-T1500  | chunks=2 | cached
08/54 | clean  | AES-T1600  | chunks=2 | cached
09/54 | clean  | AES-T1700  | chunks=2 | cached
10/54 | clean  | AES-T1800  | chunks=2 | cached
11/54 | clean  | AES-T1900  | chunks=2 | cached
12/54 | clean  | AES-T200   | chunks=2 | cached
13/54 | clean  | AES-T2000  | chunks=2 | cached
14/54 | clean  | AES-T2100  | chunks=2 | cached
15/54 | clean  | AES-T2300  | chunks=2 | computed/reused
16/54 | clean  | AES-T2400  | chunks=2 | cached
17/54 | clean  | AES-T2500  | chunks=2 | cached
18/54 | clean  | AES-T2600  | chunks=2 | cached
19/54 | clean  | AES-T2700  | chunks=2 | cached
20/54 | clean  | AES-T2800  | chunks=2 | cached
21/54 | clean  | AES-T

In [ ]:
from google.colab import files
files.download("/content/rtl_embeddings_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile
import json
from pathlib import Path

zip_path = Path("/content/rtl_embeddings_v2.zip")

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP integrity:", "PASS" if z.testzip() is None else "FAIL")

    names = z.namelist()
    print("Total files:", len(names))

    manifest_name = next(
        (n for n in names if n.endswith("manifest.json")),
        None
    )

    if manifest_name is None:
        raise FileNotFoundError("manifest.json missing from ZIP")

    manifest = json.loads(z.read(manifest_name))

    rows = manifest["designs"]
    print("Designs:", len(rows))
    print("Clean:", sum(r["label"] == 0 for r in rows))
    print("Trojan:", sum(r["label"] == 1 for r in rows))
    print("Unique source texts:", manifest["unique_source_texts"])
    print("Embedding dimension:", rows[0]["embedding_dim"])
    print("First sample:", rows[0]["benchmark"], rows[0]["embedding_path"])

    tensors = [n for n in names if n.endswith(".pt")]
    print("Tensor files:", len(tensors))

ZIP integrity: PASS
Total files: 55
Designs: 54
Clean: 27
Trojan: 27
Unique source texts: 29
Embedding dimension: 768
First sample: AES-T100 clean/AES-T100.pt
Tensor files: 54


In [ ]:
import zipfile
import json
import torch
from pathlib import Path
from collections import Counter

zip_path = Path("/content/rtl_embeddings_v2.zip")
extract_dir = Path("/content/rtl_embeddings_v2_audit")

# Extract into a separate audit directory
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# Locate the manifest
manifest_paths = list(extract_dir.rglob("manifest.json"))

if len(manifest_paths) != 1:
    raise RuntimeError(
        f"Expected exactly one manifest.json, found {len(manifest_paths)}"
    )

manifest_path = manifest_paths[0]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
rows = manifest["designs"]

print("Manifest:", manifest_path)
print("Design records:", len(rows))

# Validate tensor paths, shapes, and numerical values
errors = []
shape_counts = Counter()
validated = 0

for row in rows:
    embedding_path = Path(row["embedding_path"])

    # Handle paths stored relative to the output directory
    candidates = [
        extract_dir / embedding_path,
        manifest_path.parent / embedding_path,
        manifest_path.parent / embedding_path.name,
    ]

    tensor_path = next((p for p in candidates if p.is_file()), None)

    if tensor_path is None:
        errors.append(
            f"Missing tensor: {row.get('benchmark')} -> {row['embedding_path']}"
        )
        continue

    try:
        tensor = torch.load(
            tensor_path,
            map_location="cpu",
            weights_only=True
        )

        if not isinstance(tensor, torch.Tensor):
            errors.append(f"Not a tensor: {tensor_path}")
            continue

        if tensor.ndim != 2 or tensor.shape[1] != 768:
            errors.append(
                f"Unexpected shape: {tensor_path} -> {tuple(tensor.shape)}"
            )
            continue

        if tensor.shape[0] == 0:
            errors.append(f"Empty embedding: {tensor_path}")
            continue

        if not torch.isfinite(tensor).all():
            errors.append(f"NaN/Inf values: {tensor_path}")
            continue

        shape_counts[tuple(tensor.shape)] += 1
        validated += 1

    except Exception as exc:
        errors.append(f"Load error: {tensor_path}: {exc}")

print("\nValidated tensors:", validated)
print("Tensor shapes (rows, 768):", dict(shape_counts))
print("Errors:", len(errors))

for error in errors[:20]:
    print("ERROR:", error)

if validated == len(rows) and not errors:
    print("\nTENSOR VALIDATION: PASS")
else:
    print("\nTENSOR VALIDATION: NEEDS INVESTIGATION")

Manifest: /content/rtl_embeddings_v2_audit/rtl_embeddings_v2/manifest.json
Design records: 54

Validated tensors: 54
Tensor shapes (rows, 768): {(2, 768): 54}
Errors: 0

TENSOR VALIDATION: PASS


In [ ]:
import json
import hashlib
import torch
from pathlib import Path
from collections import defaultdict

root = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
rows = manifest["designs"]

def find_tensor(path_string):
    p = Path(path_string)
    candidates = [root / p, root / p.name, p]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(path_string)

embedding_groups = defaultdict(list)
design_info = []

for row in rows:
    path = find_tensor(row["embedding_path"])
    tensor = torch.load(path, map_location="cpu", weights_only=True).float()

    # Hash the complete embedding tensor
    digest = hashlib.sha256(
        tensor.contiguous().numpy().tobytes()
    ).hexdigest()

    embedding_groups[digest].append({
        "benchmark": row.get("benchmark"),
        "label": row.get("label"),
        "path": str(path),
        "shape": list(tensor.shape),
    })

    design_info.append({
        "benchmark": row.get("benchmark"),
        "label": row.get("label"),
        "digest": digest,
        "shape": list(tensor.shape),
    })

print("Total designs:", len(design_info))
print("Unique embedding tensors:", len(embedding_groups))
print("Duplicate groups:", sum(len(g) > 1 for g in embedding_groups.values()))

print("\nDuplicate groups:")
for digest, group in embedding_groups.items():
    if len(group) > 1:
        print("\nGroup size:", len(group))
        for item in group:
            print(
                " ",
                item["benchmark"],
                "| label =", item["label"],
                "| shape =", item["shape"]
            )

# Check whether identical embeddings cross class labels
cross_label_groups = []
for group in embedding_groups.values():
    labels = {item["label"] for item in group}
    if len(labels) > 1:
        cross_label_groups.append(group)

print("\nIdentical embedding groups spanning both labels:",
      len(cross_label_groups))

Total designs: 54
Unique embedding tensors: 20
Duplicate groups: 8

Duplicate groups:

Group size: 21
  AES-T100 | label = 0 | shape = [2, 768]
  AES-T1000 | label = 0 | shape = [2, 768]
  AES-T1100 | label = 0 | shape = [2, 768]
  AES-T1200 | label = 0 | shape = [2, 768]
  AES-T1300 | label = 0 | shape = [2, 768]
  AES-T1400 | label = 0 | shape = [2, 768]
  AES-T1500 | label = 0 | shape = [2, 768]
  AES-T1600 | label = 0 | shape = [2, 768]
  AES-T1700 | label = 0 | shape = [2, 768]
  AES-T1800 | label = 0 | shape = [2, 768]
  AES-T1900 | label = 0 | shape = [2, 768]
  AES-T200 | label = 0 | shape = [2, 768]
  AES-T2000 | label = 0 | shape = [2, 768]
  AES-T2100 | label = 0 | shape = [2, 768]
  AES-T300 | label = 0 | shape = [2, 768]
  AES-T400 | label = 0 | shape = [2, 768]
  AES-T500 | label = 0 | shape = [2, 768]
  AES-T600 | label = 0 | shape = [2, 768]
  AES-T700 | label = 0 | shape = [2, 768]
  AES-T800 | label = 0 | shape = [2, 768]
  AES-T900 | label = 0 | shape = [2, 768]

Gro

In [ ]:
import json
import torch
from pathlib import Path

root = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
rows = manifest["designs"]

print("Manifest-level keys:", list(manifest.keys()))
print("\nDesign-record keys:", list(rows[0].keys()))
print("\nFirst design record:")
for key, value in rows[0].items():
    print(f"{key}: {value}")

print("\n--- Duplicate representation metadata ---")

targets = [
    ("AES-T100", 0),
    ("AES-T1000", 0),
    ("AES-T2300", 0),
    ("AES-T1000", 1),
    ("AES-T1100", 1),
]

for benchmark, label in targets:
    matches = [
        r for r in rows
        if r.get("benchmark") == benchmark
        and r.get("label") == label
    ]

    for row in matches:
        print(f"\n{benchmark}, label={label}")
        for key, value in row.items():
            print(f"  {key}: {value}")

# Inspect one tensor's two rows
row = rows[0]
p = Path(row["embedding_path"])
candidates = [root / p, root / p.name, p]
tensor_path = next((x for x in candidates if x.is_file()), None)

if tensor_path is None:
    raise FileNotFoundError(row["embedding_path"])

tensor = torch.load(
    tensor_path, map_location="cpu", weights_only=True
).float()

print("\n--- Tensor structure ---")
print("Shape:", tuple(tensor.shape))
print("Row norms:", tensor.norm(dim=1).tolist())
print("Cosine similarity between rows:",
      torch.nn.functional.cosine_similarity(
          tensor[0:1], tensor[1:2]
      ).item())

Manifest-level keys: ['model', 'device', 'max_length', 'stride', 'batch_size', 'pooling', 'source_root', 'output_root', 'filename_and_label_inserted_into_model_input', 'testbench_exclusion_rule', 'unique_source_texts', 'designs']

Design-record keys: ['class_name', 'label', 'benchmark', 'source_text_sha256', 'included_files', 'included_file_sha256', 'excluded_testbench_files', 'num_chunks', 'embedding_dim', 'embedding_path']

First design record:
class_name: clean
label: 0
benchmark: AES-T100
source_text_sha256: d839f39773fd8ca68f99771f6fc3de3522466de8d5c99e39f81aa4add575cead
included_files: ['clean/AES-T100/aes_128.v', 'clean/AES-T100/round.v', 'clean/AES-T100/table.v']
included_file_sha256: {'clean/AES-T100/aes_128.v': '6507f2c74cb288e94ae51b2034b2669261893b39067bef3189b2f1a138a8ce52', 'clean/AES-T100/round.v': 'a9c7504563285ded97dc072a653547938b457ea1918b006651d168c61a4ef362', 'clean/AES-T100/table.v': '36171c0297f95a036285d418b9be3a0d4c4d4e5248a8f0a6133ed7956b37b157'}
excluded_test

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

root = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
manifest = json.loads((root / "manifest.json").read_text())

rows = manifest["designs"]

print("Pooling method:", manifest.get("pooling"))
print("Maximum sequence length:", manifest.get("max_length"))
print("Stride:", manifest.get("stride"))
print("Filename/label inserted:",
      manifest.get("filename_and_label_inserted_into_model_input"))

chunk_counts = defaultdict(int)
source_groups = defaultdict(list)
file_groups = defaultdict(list)

for row in rows:
    chunk_counts[row["num_chunks"]] += 1
    source_groups[row["source_text_sha256"]].append(
        (row["benchmark"], row["label"])
    )

    # Compare the actual content hashes, excluding folder-specific paths.
    file_signature = tuple(sorted(row["included_file_sha256"].values()))
    file_groups[file_signature].append(
        (row["benchmark"], row["label"])
    )

print("\nChunk-count distribution:", dict(sorted(chunk_counts.items())))
print("Unique source-text hashes:", len(source_groups))
print("Unique included-file hash sets:", len(file_groups))

print("\nSource-hash groups with multiple designs:")
for group in source_groups.values():
    if len(group) > 1:
        print(group)

print("\nIncluded-file groups with multiple designs:")
for group in file_groups.values():
    if len(group) > 1:
        print(group)

Pooling method: Attention-mask-weighted mean of final hidden states for each tokenizer chunk
Maximum sequence length: 512
Stride: 50
Filename/label inserted: False

Chunk-count distribution: {2: 54}
Unique source-text hashes: 29
Unique included-file hash sets: 29

Source-hash groups with multiple designs:
[('AES-T100', 0), ('AES-T1000', 0), ('AES-T1100', 0), ('AES-T1200', 0), ('AES-T1300', 0), ('AES-T1400', 0), ('AES-T1500', 0), ('AES-T1600', 0), ('AES-T1700', 0), ('AES-T1800', 0), ('AES-T1900', 0), ('AES-T200', 0), ('AES-T2000', 0), ('AES-T2100', 0), ('AES-T300', 0), ('AES-T400', 0), ('AES-T500', 0), ('AES-T600', 0), ('AES-T700', 0), ('AES-T800', 0), ('AES-T900', 0)]
[('AES-T2300', 0), ('AES-T2400', 0), ('AES-T2500', 0), ('AES-T2600', 0), ('AES-T2700', 0), ('AES-T2800', 0)]

Included-file groups with multiple designs:
[('AES-T100', 0), ('AES-T1000', 0), ('AES-T1100', 0), ('AES-T1200', 0), ('AES-T1300', 0), ('AES-T1400', 0), ('AES-T1500', 0), ('AES-T1600', 0), ('AES-T1700', 0), ('AES-T

In [ ]:
import json
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer

SOURCE_ROOT = Path("/content/rtl_source_data/AES")
MANIFEST_PATH = Path(
    "/content/rtl_embeddings_v2_audit/rtl_embeddings_v2/manifest.json"
)

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

window_counts = Counter()
mismatches = []

for row in manifest["designs"]:
    parts = []

    for relative_path in sorted(row["included_files"]):
        file_path = SOURCE_ROOT / relative_path

        if not file_path.is_file():
            raise FileNotFoundError(f"Missing source file: {file_path}")

        parts.append(file_path.read_text(encoding="utf-8", errors="replace"))

    source_text = "\n\n".join(parts)

    encoded = tokenizer(
        source_text,
        max_length=512,
        stride=50,
        truncation=True,
        return_overflowing_tokens=True,
        padding=False,
    )

    actual_chunks = len(encoded["input_ids"])
    recorded_chunks = row["num_chunks"]
    window_counts[actual_chunks] += 1

    if actual_chunks != recorded_chunks:
        mismatches.append({
            "benchmark": row["benchmark"],
            "label": row["label"],
            "manifest_chunks": recorded_chunks,
            "recounted_chunks": actual_chunks,
        })

print("Independent tokenizer window-count distribution:",
      dict(sorted(window_counts.items())))
print("Manifest/tokenizer mismatches:", len(mismatches))

for item in mismatches[:20]:
    print(item)

print("Chunk-count audit:",
      "PASS" if not mismatches else "INVESTIGATE")

Independent tokenizer window-count distribution: {2: 54}
Manifest/tokenizer mismatches: 0
Chunk-count audit: PASS


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/AES_graph_artifacts.zip")

with zipfile.ZipFile(zip_path, "r") as z:
    bad_file = z.testzip()
    names = z.namelist()

    print("ZIP integrity:", "PASS" if bad_file is None else f"FAIL: {bad_file}")
    print("Total files:", len(names))

    manifests = [
        n for n in names
        if n.replace("\\", "/").endswith("merged_manifest.json")
    ]
    tensors = [
        n for n in names
        if n.lower().endswith(".pt")
    ]

    print("Manifest files:", manifests)
    print("Graph tensor files:", len(tensors))

    print("\nFirst 15 archive paths:")
    for name in names[:15]:
        print(name)

ZIP integrity: PASS
Total files: 54
Manifest files: ['merged_manifest.json']
Graph tensor files: 53

First 15 archive paths:
AES\AES-T1000_clean_netlist_clean.pt
AES\AES-T1000_trojan_netlist_trojan.pt
AES\AES-T100_clean_netlist_clean.pt
AES\AES-T100_trojan_netlist_trojan.pt
AES\AES-T1100_clean_netlist_clean.pt
AES\AES-T1100_trojan_netlist_trojan.pt
AES\AES-T1200_clean_netlist_clean.pt
AES\AES-T1200_trojan_netlist_trojan.pt
AES\AES-T1300_clean_netlist_clean.pt
AES\AES-T1300_trojan_netlist_trojan.pt
AES\AES-T1400_clean_netlist_clean.pt
AES\AES-T1400_trojan_netlist_trojan.pt
AES\AES-T1500_clean_netlist_clean.pt
AES\AES-T1500_trojan_netlist_trojan.pt
AES\AES-T1600_clean_netlist_clean.pt


In [ ]:
import zipfile
import json
import re
from collections import defaultdict
from pathlib import Path

zip_path = Path("/content/AES_graph_artifacts.zip")

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    manifest_name = next(
        n for n in names
        if n.replace("\\", "/").endswith("merged_manifest.json")
    )
    manifest = json.loads(z.read(manifest_name))

    tensor_names = [
        n.replace("\\", "/")
        for n in names
        if n.lower().endswith(".pt")
    ]

print("Manifest type:", type(manifest).__name__)
if isinstance(manifest, dict):
    print("Manifest keys:", list(manifest.keys()))
    for key, value in manifest.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)} records")
            if value:
                print("First record:", value[0])
                break
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} keys")

# Detect clean/Trojan tensor pairs from filenames
found = defaultdict(set)

for name in tensor_names:
    filename = Path(name).name
    match = re.match(r"(AES-T\d+)_(clean|trojan)_netlist_", filename)

    if match:
        benchmark, label = match.groups()
        found[benchmark].add(label)

print("\nGraph tensor count:", len(tensor_names))
print("Benchmarks identified:", len(found))

print("\nMissing clean/Trojan counterparts:")
for benchmark in sorted(found):
    missing = {"clean", "trojan"} - found[benchmark]
    if missing:
        print(benchmark, "missing:", ", ".join(sorted(missing)))

print("\nTensor files not matching the expected naming pattern:")
for name in tensor_names:
    if not re.match(
        r"AES-T\d+_(clean|trojan)_netlist_",
        Path(name).name
    ):
        print(name)

Manifest type: list

Graph tensor count: 53
Benchmarks identified: 27

Missing clean/Trojan counterparts:
AES-T600 missing: trojan

Tensor files not matching the expected naming pattern:


In [ ]:
import zipfile
import json
from collections import Counter

zip_path = "/content/AES_graph_artifacts.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    manifest_name = next(
        n for n in z.namelist()
        if n.replace("\\", "/").endswith("merged_manifest.json")
    )
    manifest = json.loads(z.read(manifest_name))

print("Total manifest records:", len(manifest))
print("First record type:", type(manifest[0]).__name__)

print("\nFirst record:")
print(json.dumps(manifest[0], indent=2)[:2500])

print("\nRecords mentioning AES-T600:")
for record in manifest:
    if "T600" in json.dumps(record):
        print(json.dumps(record, indent=2)[:1500])

print("\nRecord keys:")
if manifest and isinstance(manifest[0], dict):
    print(list(manifest[0].keys()))

Total manifest records: 89
First record type: dict

First record:
{
  "graph_id": "AES/AES-T100/clean_netlist",
  "family": "AES",
  "benchmark": "AES-T100",
  "status": "clean",
  "label": 0,
  "num_nodes": 7903,
  "num_edges": 6519,
  "num_sem_chunks": 30,
  "merged_path": "AES/AES-T100_clean_netlist_clean.pt"
}

Records mentioning AES-T600:
{
  "graph_id": "AES/AES-T600/clean_netlist",
  "family": "AES",
  "benchmark": "AES-T600",
  "status": "clean",
  "label": 0,
  "num_nodes": 7903,
  "num_edges": 6519,
  "num_sem_chunks": 30,
  "merged_path": "AES/AES-T600_clean_netlist_clean.pt"
}
{
  "graph_id": "RS232/RS232-T600/standard_netlist",
  "family": "RS232",
  "benchmark": "RS232-T600",
  "status": "trojan",
  "label": 1,
  "num_nodes": 106,
  "num_edges": 132,
  "num_sem_chunks": 12,
  "merged_path": "RS232/RS232-T600_standard_netlist_trojan.pt"
}

Record keys:
['graph_id', 'family', 'benchmark', 'status', 'label', 'num_nodes', 'num_edges', 'num_sem_chunks', 'merged_path']


In [ ]:
import zipfile
import json
from pathlib import Path
from collections import defaultdict

zip_path = Path("/content/AES_graph_artifacts.zip")
rtl_manifest_path = Path(
    "/content/rtl_embeddings_v2_audit/rtl_embeddings_v2/manifest.json"
)

with zipfile.ZipFile(zip_path, "r") as z:
    names = [n.replace("\\", "/") for n in z.namelist()]
    manifest_name = next(
        n for n in names if n.endswith("merged_manifest.json")
    )
    graph_manifest = json.loads(z.read(manifest_name))

rtl_manifest = json.loads(
    rtl_manifest_path.read_text(encoding="utf-8")
)

graph_tensors = {
    Path(n).name
    for n in names
    if n.lower().endswith(".pt")
}

# Index AES graph records by benchmark and class label
graph_records = {}
for row in graph_manifest:
    if row.get("family") != "AES":
        continue
    key = (row["benchmark"], row["label"])
    graph_records[key] = row

print("RTL records:", len(rtl_manifest["designs"]))
print("AES graph manifest records:", len(graph_records))
print("Graph tensors in archive:", len(graph_tensors))

missing = []
matched = []

for row in rtl_manifest["designs"]:
    benchmark = row["benchmark"]
    label = row["label"]

    # Exclude AES-T600 from both classes for this experiment
    if benchmark == "AES-T600":
        continue

    key = (benchmark, label)
    graph_row = graph_records.get(key)

    if graph_row is None:
        missing.append((benchmark, label, "No matching graph manifest record"))
        continue

    tensor_name = Path(graph_row["merged_path"]).name

    if tensor_name not in graph_tensors:
        missing.append((benchmark, label, f"Missing tensor: {tensor_name}"))
        continue

    matched.append((benchmark, label, tensor_name))

print("\nMatched samples:", len(matched))
print("Missing samples:", len(missing))

for item in missing:
    print("MISSING:", item)

print("\nMatched clean:", sum(label == 0 for _, label, _ in matched))
print("Matched Trojan:", sum(label == 1 for _, label, _ in matched))

RTL records: 54
AES graph manifest records: 53
Graph tensors in archive: 53

Matched samples: 52
Missing samples: 0

Matched clean: 26
Matched Trojan: 26


In [ ]:

import zipfile
import io
import json
import torch
from collections import Counter

zip_path = "/content/AES_graph_artifacts.zip"

def describe(obj, path="root", depth=0):
    if depth > 3:
        return [f"{path}: nested structure omitted"]

    if isinstance(obj, torch.Tensor):
        finite = "N/A"
        if obj.is_floating_point() or obj.is_complex():
            finite = bool(torch.isfinite(obj).all().item())
        return [
            f"{path}: Tensor shape={tuple(obj.shape)}, "
            f"dtype={obj.dtype}, finite={finite}"
        ]

    if isinstance(obj, dict):
        lines = [f"{path}: dict keys={list(obj.keys())}"]
        for k, v in list(obj.items())[:20]:
            lines.extend(describe(v, f"{path}[{k!r}]", depth + 1))
        return lines

    if isinstance(obj, (list, tuple)):
        lines = [f"{path}: {type(obj).__name__}, len={len(obj)}"]
        for i, v in enumerate(obj[:10]):
            lines.extend(describe(v, f"{path}[{i}]", depth + 1))
        return lines

    return [f"{path}: object type={type(obj).__name__}"]

with zipfile.ZipFile(zip_path, "r") as z:
    original_names = z.namelist()

    manifest_original = next(
        n for n in original_names
        if n.replace("\\", "/").endswith("merged_manifest.json")
    )
    manifest = json.loads(z.read(manifest_original))

    tensor_names = [
        n for n in original_names if n.lower().endswith(".pt")
    ]

    results, errors = [], []

    for original_name in tensor_names:
        try:
            obj = torch.load(
                io.BytesIO(z.read(original_name)),
                map_location="cpu",
                weights_only=True
            )
            display_name = original_name.replace("\\", "/")
            results.append(
                (display_name, type(obj).__name__, describe(obj))
            )
        except Exception as e:
            errors.append((
                original_name.replace("\\", "/"),
                type(e).__name__,
                str(e)[:300]
            ))

print("TOTAL GRAPH FILES:", len(tensor_names))
print("SUCCESSFULLY LOADED:", len(results))
print("LOAD ERRORS:", len(errors))

print("\n========== FIRST 3 GRAPH STRUCTURES ==========")
for name, obj_type, summary in results[:3]:
    print(f"\nFILE: {name}")
    print("TOP-LEVEL TYPE:", obj_type)
    for line in summary:
        print(" ", line)

print("\n========== LOAD ERRORS ==========")
for item in errors:
    print(item)

aes_records = [
    r for r in manifest if r.get("family") == "AES"
]
print("\nAES manifest records:", len(aes_records))
print("Labels:", Counter(r["label"] for r in aes_records))


TOTAL GRAPH FILES: 53
SUCCESSFULLY LOADED: 0
LOAD ERRORS: 53

========== FIRST 3 GRAPH STRUCTURES ==========

========== LOAD ERRORS ==========
('AES/AES-T1000_clean_netlist_clean.pt', 'UnpicklingError', 'Weights only load failed. This file can still be loaded, to do so you have two options, \x1bdo those steps only if you trust the source of the checkpoint\x1b. \n\t(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.l')
('AES/AES-T1000_trojan_netlist_trojan.pt', 'UnpicklingError', 'Weights only load failed. This file can still be loaded, to do so you have two options, \x1bdo those steps only if you trust the source of the checkpoint\x1b. \n\t(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.l')
('AES/AES-T100_clean_netlist_clean.pt', 'UnpicklingError', 'Weights only load failed. This file can still be loaded, to do

In [ ]:

%pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.7 MB/s eta 0:00:00


In [ ]:
import zipfile
import io
import torch
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

zip_path = "/content/AES_graph_artifacts.zip"
target = "AES/AES-T100_clean_netlist_clean.pt"

with zipfile.ZipFile(zip_path, "r") as z:
    original_name = next(
        n for n in z.namelist()
        if n.replace("\\", "/") == target
    )
    raw = z.read(original_name)

with torch.serialization.safe_globals([Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]):
    graph = torch.load(
        io.BytesIO(raw),
        map_location="cpu",
        weights_only=True
    )

print("LOAD SUCCESSFUL")
print("Number of nodes:", graph.num_nodes)
print("Number of edges:", graph.num_edges)
for field in ("x", "edge_index", "edge_attr", "y"):
    value = getattr(graph, field, None)
    print(f"{field}:", None if value is None else (
        tuple(value.shape) if isinstance(value, torch.Tensor) else repr(value)
    ))

LOAD SUCCESSFUL
Number of nodes: 7903
Number of edges: 6519
x: (7903, 20)
edge_index: (2, 6519)
edge_attr: None
y: (1,)


In [ ]:

import zipfile
import io
import json
import torch
from collections import Counter
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

zip_path = "/content/AES_graph_artifacts.zip"

safe_classes = [Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]

with zipfile.ZipFile(zip_path, "r") as z:
    original_names = z.namelist()

    manifest_name = next(
        n for n in original_names
        if n.replace("\\", "/").endswith("merged_manifest.json")
    )
    manifest = json.loads(z.read(manifest_name))

    manifest_by_path = {
        r["merged_path"].replace("\\", "/"): r
        for r in manifest
        if r.get("family") == "AES"
    }

    tensor_names = [
        n for n in original_names if n.lower().endswith(".pt")
    ]

    passed, failed = [], []

    with torch.serialization.safe_globals(safe_classes):
        for name in tensor_names:
            display_name = name.replace("\\", "/")
            record = manifest_by_path.get(display_name)

            try:
                graph = torch.load(
                    io.BytesIO(z.read(name)),
                    map_location="cpu",
                    weights_only=True
                )

                issues = []

                if not isinstance(graph, Data):
                    issues.append(f"Unexpected type: {type(graph)}")

                x = getattr(graph, "x", None)
                edge_index = getattr(graph, "edge_index", None)
                y = getattr(graph, "y", None)

                if not isinstance(x, torch.Tensor):
                    issues.append("Missing/non-tensor x")
                else:
                    if x.ndim != 2:
                        issues.append(f"x shape invalid: {tuple(x.shape)}")
                    if not torch.isfinite(x).all().item():
                        issues.append("x contains NaN/Inf")

                if not isinstance(edge_index, torch.Tensor):
                    issues.append("Missing/non-tensor edge_index")
                else:
                    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
                        issues.append(
                            f"edge_index shape invalid: {tuple(edge_index.shape)}"
                        )
                    elif edge_index.numel() > 0 and isinstance(x, torch.Tensor):
                        if edge_index.min().item() < 0:
                            issues.append("Negative edge index")
                        if edge_index.max().item() >= x.shape[0]:
                            issues.append("Edge index exceeds node count")

                if record is None:
                    issues.append("No matching manifest record")
                elif isinstance(x, torch.Tensor):
                    if x.shape[0] != record.get("num_nodes"):
                        issues.append(
                            f"Node mismatch: x={x.shape[0]}, "
                            f"manifest={record.get('num_nodes')}"
                        )
                    if edge_index is not None and edge_index.shape[-1] != record.get("num_edges"):
                        issues.append(
                            f"Edge mismatch: edge_index={edge_index.shape[-1]}, "
                            f"manifest={record.get('num_edges')}"
                        )

                if isinstance(y, torch.Tensor):
                    if y.numel() != 1:
                        issues.append(f"Unexpected y shape: {tuple(y.shape)}")
                    elif record is not None and y.item() != record["label"]:
                        issues.append(
                            f"Label mismatch: y={y.item()}, "
                            f"manifest={record['label']}"
                        )

                info = {
                    "file": display_name,
                    "nodes": getattr(graph, "num_nodes", None),
                    "edges": getattr(graph, "num_edges", None),
                    "x_shape": None if x is None else tuple(x.shape),
                    "y": None if y is None else y.tolist(),
                    "manifest_label": None if record is None else record["label"],
                    "issues": issues,
                }

                if issues:
                    failed.append(info)
                else:
                    passed.append(info)

            except Exception as e:
                failed.append({
                    "file": display_name,
                    "issues": [f"{type(e).__name__}: {str(e)[:250]}"]
                })

print("Total graph files:", len(tensor_names))
print("Passed:", len(passed))
print("Failed:", len(failed))

print("\nNode feature dimensions:")
print(Counter(str(r["x_shape"]) for r in passed))

print("\nFirst 5 passed graphs:")
for r in passed[:5]:
    print(r)

print("\nFailures:")
for r in failed:
    print(r)


Total graph files: 53
Passed: 53
Failed: 0

Node feature dimensions:
Counter({'(7903, 20)': 27, '(7705, 20)': 9, '(7905, 20)': 2, '(7907, 20)': 1, '(7716, 20)': 1, '(8076, 20)': 1, '(8289, 20)': 1, '(8057, 20)': 1, '(8276, 20)': 1, '(7736, 20)': 1, '(7916, 20)': 1, '(7917, 20)': 1, '(7866, 20)': 1, '(7931, 20)': 1, '(7892, 20)': 1, '(7845, 20)': 1, '(8017, 20)': 1, '(8229, 20)': 1})

First 5 passed graphs:
{'file': 'AES/AES-T1000_clean_netlist_clean.pt', 'nodes': 7903, 'edges': 6519, 'x_shape': (7903, 20), 'y': [0], 'manifest_label': 0, 'issues': []}
{'file': 'AES/AES-T1000_trojan_netlist_trojan.pt', 'nodes': 7907, 'edges': 6719, 'x_shape': (7907, 20), 'y': [1], 'manifest_label': 1, 'issues': []}
{'file': 'AES/AES-T100_clean_netlist_clean.pt', 'nodes': 7903, 'edges': 6519, 'x_shape': (7903, 20), 'y': [0], 'manifest_label': 0, 'issues': []}
{'file': 'AES/AES-T100_trojan_netlist_trojan.pt', 'nodes': 7716, 'edges': 6530, 'x_shape': (7716, 20), 'y': [1], 'manifest_label': 1, 'issues': []}


In [ ]:

import json
from pathlib import Path

rtl_root = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
manifest_path = rtl_root / "manifest.json"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print("Manifest top-level keys:", list(manifest.keys()))

designs = manifest.get("designs", [])
print("Number of RTL designs:", len(designs))

print("\n========== FIRST RTL RECORD ==========")
if designs:
    print(json.dumps(designs[0], indent=2)[:5000])

print("\n========== RTL EMBEDDING FILES ==========")
for path in sorted(rtl_root.rglob("*")):
    if path.is_file() and path.name != "manifest.json":
        print(path.relative_to(rtl_root), path.stat().st_size, "bytes")


Manifest top-level keys: ['model', 'device', 'max_length', 'stride', 'batch_size', 'pooling', 'source_root', 'output_root', 'filename_and_label_inserted_into_model_input', 'testbench_exclusion_rule', 'unique_source_texts', 'designs']
Number of RTL designs: 54

========== FIRST RTL RECORD ==========
{
  "class_name": "clean",
  "label": 0,
  "benchmark": "AES-T100",
  "source_text_sha256": "d839f39773fd8ca68f99771f6fc3de3522466de8d5c99e39f81aa4add575cead",
  "included_files": [
    "clean/AES-T100/aes_128.v",
    "clean/AES-T100/round.v",
    "clean/AES-T100/table.v"
  ],
  "included_file_sha256": {
    "clean/AES-T100/aes_128.v": "6507f2c74cb288e94ae51b2034b2669261893b39067bef3189b2f1a138a8ce52",
    "clean/AES-T100/round.v": "a9c7504563285ded97dc072a653547938b457ea1918b006651d168c61a4ef362",
    "clean/AES-T100/table.v": "36171c0297f95a036285d418b9be3a0d4c4d4e5248a8f0a6133ed7956b37b157"
  },
  "excluded_testbench_files": [
    "clean/AES-T100/test_aes_128.v"
  ],
  "num_chunks": 2,
  

In [ ]:

import json
import zipfile
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import torch
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

ZIP_PATH = Path("/content/AES_graph_artifacts.zip")
RTL_ROOT = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
RTL_MANIFEST = RTL_ROOT / "manifest.json"
OUT_PATH = Path("/content/paired_aes_manifest_v1.json")

rtl_manifest = json.loads(RTL_MANIFEST.read_text(encoding="utf-8"))
rtl_records = rtl_manifest["designs"]

safe_classes = [Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    original_names = z.namelist()
    normalized_to_original = {
        n.replace("\\", "/"): n for n in original_names
    }

    manifest_name = next(
        n for n in original_names
        if n.replace("\\", "/").endswith("merged_manifest.json")
    )
    graph_manifest = json.loads(z.read(manifest_name))

    graph_records = {}
    for r in graph_manifest:
        if r.get("family") != "AES":
            continue
        graph_records[(r["benchmark"], int(r["label"]))] = r

    graph_tensors = {
        n.replace("\\", "/")
        for n in original_names
        if n.lower().endswith(".pt")
    }

    paired = []
    problems = []

    with torch.serialization.safe_globals(safe_classes):
        for r in rtl_records:
            benchmark = r["benchmark"]
            label = int(r["label"])

            # Exclude both AES-T600 classes from this experiment
            if benchmark == "AES-T600":
                continue

            if label != (0 if r["class_name"] == "clean" else 1):
                problems.append(
                    f"{benchmark}: RTL class_name/label mismatch"
                )
                continue

            emb_path = RTL_ROOT / r["embedding_path"]
            if not emb_path.is_file():
                problems.append(
                    f"{benchmark} label={label}: missing embedding {emb_path}"
                )
                continue

            try:
                embedding = torch.load(
                    emb_path, map_location="cpu", weights_only=True
                )
            except Exception as e:
                problems.append(
                    f"{benchmark} label={label}: embedding load failed: {e}"
                )
                continue

            if not isinstance(embedding, torch.Tensor):
                problems.append(
                    f"{benchmark} label={label}: embedding is "
                    f"{type(embedding).__name__}, not a tensor"
                )
                continue

            if embedding.ndim != 2 or embedding.shape[-1] != 768:
                problems.append(
                    f"{benchmark} label={label}: unexpected embedding "
                    f"shape {tuple(embedding.shape)}"
                )
                continue

            if not torch.isfinite(embedding).all().item():
                problems.append(
                    f"{benchmark} label={label}: embedding has NaN/Inf"
                )
                continue

            graph_r = graph_records.get((benchmark, label))
            if graph_r is None:
                problems.append(
                    f"{benchmark} label={label}: no graph manifest record"
                )
                continue

            graph_path = graph_r["merged_path"].replace("\\", "/")
            if graph_path not in graph_tensors:
                problems.append(
                    f"{benchmark} label={label}: missing graph file "
                    f"{graph_path}"
                )
                continue

            try:
                graph = torch.load(
                    __import__("io").BytesIO(
                        z.read(normalized_to_original[graph_path])
                    ),
                    map_location="cpu",
                    weights_only=True
                )
            except Exception as e:
                problems.append(
                    f"{benchmark} label={label}: graph load failed: {e}"
                )
                continue

            if not isinstance(graph, Data):
                problems.append(
                    f"{benchmark} label={label}: graph is not PyG Data"
                )
                continue

            if graph.x is None or graph.edge_index is None:
                problems.append(
                    f"{benchmark} label={label}: missing x/edge_index"
                )
                continue

            if graph.y is None or graph.y.numel() != 1:
                problems.append(
                    f"{benchmark} label={label}: missing/invalid graph label"
                )
                continue

            if int(graph.y.item()) != label:
                problems.append(
                    f"{benchmark}: graph label {graph.y.item()} "
                    f"does not match RTL label {label}"
                )
                continue

            paired.append({
                "benchmark": benchmark,
                "label": label,
                "class_name": r["class_name"],
                "rtl_embedding_path": r["embedding_path"],
                "rtl_embedding_shape": list(embedding.shape),
                "source_text_sha256": r["source_text_sha256"],
                "included_files": r["included_files"],
                "included_file_sha256": r["included_file_sha256"],
                "graph_archive_path": graph_path,
                "graph_num_nodes": int(graph.num_nodes),
                "graph_num_edges": int(graph.num_edges),
                "graph_node_feature_dim": int(graph.x.shape[1]),
            })

# Audit exact RTL source duplicates within the selected paired set
hash_groups = defaultdict(list)
for r in paired:
    hash_groups[r["source_text_sha256"]].append(
        (r["benchmark"], r["label"])
    )

duplicate_groups = {
    h: samples for h, samples in hash_groups.items()
    if len(samples) > 1
}

print("========== PAIRING AUDIT ==========")
print("RTL records:", len(rtl_records))
print("Paired samples:", len(paired))
print("Labels:", dict(Counter(r["label"] for r in paired)))
print("Problems:", len(problems))
for p in problems:
    print("PROBLEM:", p)

print("\nUnique RTL source hashes:", len(hash_groups))
print("Duplicate source groups:", len(duplicate_groups))
print("\nDuplicate groups:")
for samples in duplicate_groups.values():
    print(samples)

if len(paired) == 52 and not problems:
    output = {
        "description": "Paired AES graph and RTL embedding audit",
        "excluded_benchmark": "AES-T600",
        "num_samples": len(paired),
        "label_counts": dict(Counter(r["label"] for r in paired)),
        "num_unique_rtl_source_hashes": len(hash_groups),
        "duplicate_source_groups": [
            {
                "samples": [
                    {"benchmark": b, "label": y}
                    for b, y in samples
                ]
            }
            for samples in duplicate_groups.values()
        ],
        "samples": sorted(
            paired, key=lambda r: (r["benchmark"], r["label"])
        )
    }
    OUT_PATH.write_text(json.dumps(output, indent=2), encoding="utf-8")
    print("\nAUDIT PASS")
    print("Saved manifest:", OUT_PATH)
else:
    print("\nAUDIT NOT PASSED — do not train yet.")


========== PAIRING AUDIT ==========
RTL records: 54
Paired samples: 52
Labels: {0: 26, 1: 26}
Problems: 0

Unique RTL source hashes: 28
Duplicate source groups: 2

Duplicate groups:
[('AES-T100', 0), ('AES-T1000', 0), ('AES-T1100', 0), ('AES-T1200', 0), ('AES-T1300', 0), ('AES-T1400', 0), ('AES-T1500', 0), ('AES-T1600', 0), ('AES-T1700', 0), ('AES-T1800', 0), ('AES-T1900', 0), ('AES-T200', 0), ('AES-T2000', 0), ('AES-T2100', 0), ('AES-T300', 0), ('AES-T400', 0), ('AES-T500', 0), ('AES-T700', 0), ('AES-T800', 0), ('AES-T900', 0)]
[('AES-T2300', 0), ('AES-T2400', 0), ('AES-T2500', 0), ('AES-T2600', 0), ('AES-T2700', 0), ('AES-T2800', 0)]

AUDIT PASS
Saved manifest: /content/paired_aes_manifest_v1.json


In [ ]:

import json
from pathlib import Path
from collections import defaultdict, Counter

manifest_path = Path("/content/paired_aes_manifest_v1.json")
output_path = Path("/content/paired_aes_group_folds_v1.json")

data = json.loads(manifest_path.read_text(encoding="utf-8"))
samples = data["samples"]

# Map each benchmark to its clean RTL source hash.
clean_hash_by_benchmark = {}
labels_by_benchmark = defaultdict(set)

for s in samples:
    labels_by_benchmark[s["benchmark"]].add(s["label"])
    if s["label"] == 0:
        clean_hash_by_benchmark[s["benchmark"]] = (
            s["source_text_sha256"]
        )

# Every benchmark must have exactly one clean and one Trojan sample.
bad_pairs = {
    b: sorted(labels)
    for b, labels in labels_by_benchmark.items()
    if labels != {0, 1}
}
assert not bad_pairs, f"Incomplete benchmark pairs: {bad_pairs}"

# Group benchmarks by their clean RTL source hash.
groups = defaultdict(list)
for benchmark, source_hash in clean_hash_by_benchmark.items():
    groups[source_hash].append(benchmark)

group_list = sorted(
    [sorted(v) for v in groups.values()],
    key=len,
    reverse=True
)

print("Clean RTL source-group sizes:", [len(g) for g in group_list])
assert len(group_list) == 2, (
    "Expected the two known clean source groups; inspect audit output."
)

folds = []

# Each fold tests one entire clean-source group and its paired Trojan samples.
for fold_id, test_benchmarks in enumerate(group_list, start=1):
    test_set = set(test_benchmarks)
    train_benchmarks = sorted(
        set(clean_hash_by_benchmark) - test_set
    )

    train_samples = [
        s for s in samples if s["benchmark"] in train_benchmarks
    ]
    test_samples = [
        s for s in samples if s["benchmark"] in test_set
    ]

    train_hashes = {s["source_text_sha256"] for s in train_samples}
    test_hashes = {s["source_text_sha256"] for s in test_samples}
    overlap = train_hashes & test_hashes

    fold = {
        "fold": fold_id,
        "train_benchmarks": train_benchmarks,
        "test_benchmarks": sorted(test_set),
        "train_samples": train_samples,
        "test_samples": test_samples,
        "train_label_counts": dict(
            Counter(s["label"] for s in train_samples)
        ),
        "test_label_counts": dict(
            Counter(s["label"] for s in test_samples)
        ),
        "source_hash_overlap_count": len(overlap),
        "source_hash_overlap": sorted(overlap),
    }

    assert not overlap, f"Source leakage in fold {fold_id}: {overlap}"
    assert len(train_samples) + len(test_samples) == 52
    assert set(fold["train_label_counts"]) == {0, 1}
    assert set(fold["test_label_counts"]) == {0, 1}

    folds.append(fold)

    print(f"\nFOLD {fold_id}")
    print("Train benchmarks:", len(train_benchmarks))
    print("Test benchmarks:", len(test_set))
    print("Train samples:", len(train_samples),
          fold["train_label_counts"])
    print("Test samples:", len(test_samples),
          fold["test_label_counts"])
    print("Source-hash overlap:", len(overlap))

output = {
    "method": "two-fold clean-source-group-held-out evaluation",
    "seed": None,
    "note": (
        "Small dataset; only two distinct clean RTL source groups. "
        "Report each fold separately and pooled out-of-fold results. "
        "Do not use test-fold metrics for model selection."
    ),
    "folds": folds,
}

output_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
print("\nSPLIT AUDIT PASS")
print("Saved:", output_path)


Clean RTL source-group sizes: [20, 6]

FOLD 1
Train benchmarks: 6
Test benchmarks: 20
Train samples: 12 {0: 6, 1: 6}
Test samples: 40 {0: 20, 1: 20}
Source-hash overlap: 0

FOLD 2
Train benchmarks: 20
Test benchmarks: 6
Train samples: 40 {0: 20, 1: 20}
Test samples: 12 {0: 6, 1: 6}
Source-hash overlap: 0

SPLIT AUDIT PASS
Saved: /content/paired_aes_group_folds_v1.json


In [ ]:

import json
from pathlib import Path
from collections import Counter

import torch

manifest_path = Path("/content/paired_aes_manifest_v1.json")
data = json.loads(manifest_path.read_text(encoding="utf-8"))
rtl_root = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")

shapes = Counter()
errors = []
embedding_norms = []
loaded = []

for sample in data["samples"]:
    path = rtl_root / sample["rtl_embedding_path"]

    try:
        emb = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if not isinstance(emb, torch.Tensor):
            raise TypeError(f"Expected Tensor, got {type(emb)}")

        if emb.ndim != 2 or emb.shape[-1] != 768:
            raise ValueError(f"Unexpected shape: {tuple(emb.shape)}")

        if not torch.isfinite(emb).all().item():
            raise ValueError("Embedding contains NaN/Inf")

        shapes[tuple(emb.shape)] += 1
        embedding_norms.append(float(emb.norm(dim=-1).mean()))
        loaded.append(sample)

    except Exception as e:
        errors.append((sample["benchmark"], sample["label"], str(e)))

print("Expected paired samples:", len(data["samples"]))
print("Successfully loaded:", len(loaded))
print("Errors:", len(errors))
print("Embedding shapes:", dict(shapes))

if embedding_norms:
    print("Mean per-sample chunk-vector norm:",
          sum(embedding_norms) / len(embedding_norms))

for error in errors:
    print("ERROR:", error)

assert len(loaded) == 52 and not errors, (
    "Fix embedding loading errors before training."
)

print("\nRTL EMBEDDING AUDIT PASS")


Expected paired samples: 52
Successfully loaded: 52
Errors: 0
Embedding shapes: {(2, 768): 52}
Mean per-sample chunk-vector norm: 18.346288717710056

RTL EMBEDDING AUDIT PASS


In [ ]:

import json
from pathlib import Path

import numpy as np
import torch
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

RTL_ROOT = Path("/content/rtl_embeddings_v2_audit/rtl_embeddings_v2")
FOLD_PATH = Path("/content/paired_aes_group_folds_v1.json")

with open(FOLD_PATH, "r", encoding="utf-8") as f:
    split_data = json.load(f)

def load_mean_embedding(sample):
    path = RTL_ROOT / sample["rtl_embedding_path"]
    emb = torch.load(path, map_location="cpu", weights_only=True)

    if not isinstance(emb, torch.Tensor):
        raise TypeError(f"Not a tensor: {path}")
    if emb.ndim != 2 or emb.shape != (2, 768):
        raise ValueError(f"Unexpected shape {tuple(emb.shape)}: {path}")
    if not torch.isfinite(emb).all().item():
        raise ValueError(f"NaN/Inf in {path}")

    # Two chunk vectors -> one 768-dimensional design vector
    return emb.mean(dim=0).numpy().astype(np.float32)

def build_xy(samples):
    X = np.stack([load_mean_embedding(s) for s in samples])
    y = np.array([int(s["label"]) for s in samples])
    return X, y

all_predictions = []

for fold in split_data["folds"]:
    fold_id = fold["fold"]
    train_samples = fold["train_samples"]
    test_samples = fold["test_samples"]

    X_train, y_train = build_xy(train_samples)
    X_test, y_test = build_xy(test_samples)

    # Fixed pipeline: scaling is fitted only on training data.
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42,
        ),
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n{'=' * 18} FOLD {fold_id} {'=' * 18}")
    print("Train size:", len(y_train), "Test size:", len(y_test))
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Balanced accuracy:", round(
        balanced_accuracy_score(y_test, y_pred), 4
    ))
    print("Macro-F1:", round(
        f1_score(y_test, y_pred, average="macro", zero_division=0), 4
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
    print("\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["Clean", "Trojan"],
        zero_division=0,
    ))

    for sample, true_label, pred_label in zip(
        test_samples, y_test, y_pred
    ):
        all_predictions.append({
            "fold": fold_id,
            "benchmark": sample["benchmark"],
            "true_label": int(true_label),
            "predicted_label": int(pred_label),
        })

# Every sample should be tested exactly once across the two folds.
assert len(all_predictions) == 52
assert len({
    (p["benchmark"], p["true_label"]) for p in all_predictions
}) == 52

out_path = Path("/content/rtl_only_group_cv_predictions.json")
out_path.write_text(
    json.dumps(all_predictions, indent=2),
    encoding="utf-8",
)

print("\nSaved out-of-fold predictions:", out_path)
print("Total out-of-fold predictions:", len(all_predictions))



================== FOLD 1 ==================
Train size: 12 Test size: 40
Accuracy: 0.9
Balanced accuracy: 0.9
Macro-F1: 0.899
Confusion matrix [[TN, FP], [FN, TP]]:
[[20  0]
 [ 4 16]]

Classification report:
              precision    recall  f1-score   support

       Clean       0.83      1.00      0.91        20
      Trojan       1.00      0.80      0.89        20

    accuracy                           0.90        40
   macro avg       0.92      0.90      0.90        40
weighted avg       0.92      0.90      0.90        40


================== FOLD 2 ==================
Train size: 40 Test size: 12
Accuracy: 0.8333
Balanced accuracy: 0.8333
Macro-F1: 0.8286
Confusion matrix [[TN, FP], [FN, TP]]:
[[6 0]
 [2 4]]

Classification report:
              precision    recall  f1-score   support

       Clean       0.75      1.00      0.86         6
      Trojan       1.00      0.67      0.80         6

    accuracy                           0.83        12
   macro avg       0.88      0.8

In [ ]:

import io
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report
)

SEED = 42
EPOCHS = 60
BATCH_SIZE = 4
ZIP_PATH = "/content/AES_graph_artifacts.zip"
FOLD_PATH = "/content/paired_aes_group_folds_v1.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

with open(FOLD_PATH, "r", encoding="utf-8") as f:
    split_data = json.load(f)

safe_classes = [Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]

# Load graph files once; keep the validated graph objects in memory.
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    original_names = z.namelist()
    path_lookup = {
        n.replace("\\", "/"): n for n in original_names
    }

    graph_cache = {}

    with torch.serialization.safe_globals(safe_classes):
        for fold in split_data["folds"]:
            for sample in fold["train_samples"] + fold["test_samples"]:
                path = sample["graph_archive_path"]

                if path in graph_cache:
                    continue

                graph = torch.load(
                    io.BytesIO(z.read(path_lookup[path])),
                    map_location="cpu",
                    weights_only=True
                )

                graph.x = graph.x.float()
                graph.edge_index = graph.edge_index.long()
                graph.y = torch.tensor(
                    [int(sample["label"])], dtype=torch.long
                )

                graph_cache[path] = graph

print("Cached graphs:", len(graph_cache))

class GCNClassifier(nn.Module):
    def __init__(self, in_channels=20, hidden_channels=32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, 2)
        self.dropout = 0.3

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, data.edge_index)
        x = F.relu(x)

        mean_pool = global_mean_pool(x, data.batch)
        max_pool = global_max_pool(x, data.batch)
        graph_embedding = torch.cat([mean_pool, max_pool], dim=1)

        x = F.relu(self.fc1(graph_embedding))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc2(x)

all_predictions = []

for fold in split_data["folds"]:
    fold_id = fold["fold"]

    train_graphs = [
        graph_cache[s["graph_archive_path"]]
        for s in fold["train_samples"]
    ]
    test_graphs = [
        graph_cache[s["graph_archive_path"]]
        for s in fold["test_samples"]
    ]

    # Reset seed and model parameters for reproducibility.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_loader = DataLoader(
        train_graphs, batch_size=BATCH_SIZE, shuffle=True
    )
    test_loader = DataLoader(
        test_graphs, batch_size=BATCH_SIZE, shuffle=False
    )

    model = GCNClassifier().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.001, weight_decay=5e-4
    )

    # Fixed epoch count: no test-set early stopping or tuning.
    for epoch in range(EPOCHS):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = F.cross_entropy(logits, batch.y.view(-1))
            loss.backward()
            optimizer.step()

    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=1)

            y_true.extend(batch.y.view(-1).cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    print(f"\n{'=' * 15} FOLD {fold_id} {'=' * 15}")
    print("Train graphs:", len(train_graphs), "Test graphs:", len(test_graphs))
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(
        balanced_accuracy_score(y_true, y_pred), 4
    ))
    print("Macro-F1:", round(
        f1_score(y_true, y_pred, average="macro", zero_division=0), 4
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print(classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["Clean", "Trojan"],
        zero_division=0
    ))

    for sample, true_label, pred_label in zip(
        fold["test_samples"], y_true, y_pred
    ):
        all_predictions.append({
            "fold": fold_id,
            "benchmark": sample["benchmark"],
            "true_label": int(true_label),
            "predicted_label": int(pred_label)
        })

assert len(all_predictions) == 52

out_path = Path("/content/gnn_only_group_cv_predictions.json")
out_path.write_text(
    json.dumps(all_predictions, indent=2), encoding="utf-8"
)

print("\nSaved predictions:", out_path)
print("Total out-of-fold predictions:", len(all_predictions))


Device: cuda
Cached graphs: 52


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(



=============== FOLD 1 ===============
Train graphs: 12 Test graphs: 40
Accuracy: 0.7
Balanced accuracy: 0.7
Macro-F1: 0.6703
Confusion matrix [[TN, FP], [FN, TP]]:
[[20  0]
 [12  8]]
              precision    recall  f1-score   support

       Clean       0.62      1.00      0.77        20
      Trojan       1.00      0.40      0.57        20

    accuracy                           0.70        40
   macro avg       0.81      0.70      0.67        40
weighted avg       0.81      0.70      0.67        40



/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(



=============== FOLD 2 ===============
Train graphs: 40 Test graphs: 12
Accuracy: 0.5
Balanced accuracy: 0.5
Macro-F1: 0.3333
Confusion matrix [[TN, FP], [FN, TP]]:
[[6 0]
 [6 0]]
              precision    recall  f1-score   support

       Clean       0.50      1.00      0.67         6
      Trojan       0.00      0.00      0.00         6

    accuracy                           0.50        12
   macro avg       0.25      0.50      0.33        12
weighted avg       0.25      0.50      0.33        12


Saved predictions: /content/gnn_only_group_cv_predictions.json
Total out-of-fold predictions: 52


In [ ]:

import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix
)

rtl_path = Path("/content/rtl_only_group_cv_predictions.json")
gnn_path = Path("/content/gnn_only_group_cv_predictions.json")

rtl = json.loads(rtl_path.read_text(encoding="utf-8"))
gnn = json.loads(gnn_path.read_text(encoding="utf-8"))

def key(row):
    return (row["fold"], row["benchmark"], row["true_label"])

rtl_map = {key(r): r for r in rtl}
gnn_map = {key(r): r for r in gnn}

assert rtl_map.keys() == gnn_map.keys(), "Prediction sample mismatch"

rows = []
for k in sorted(rtl_map):
    r = rtl_map[k]
    g = gnn_map[k]
    rows.append({
        "fold": k[0],
        "benchmark": k[1],
        "true": k[2],
        "rtl": r["predicted_label"],
        "gnn": g["predicted_label"],
    })

y = [r["true"] for r in rows]

rules = {
    "RTL only": [r["rtl"] for r in rows],
    "GNN only": [r["gnn"] for r in rows],
    "OR fusion (Trojan if either predicts Trojan)": [
        int(r["rtl"] == 1 or r["gnn"] == 1) for r in rows
    ],
    "AND fusion (Trojan if both predict Trojan)": [
        int(r["rtl"] == 1 and r["gnn"] == 1) for r in rows
    ],
}

for name, pred in rules.items():
    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y, pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y, pred), 4))
    print("Macro-F1:", round(
        f1_score(y, pred, average="macro", zero_division=0), 4
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y, pred, labels=[0, 1]))

rtl_missed = {
    r["benchmark"] for r in rows
    if r["true"] == 1 and r["rtl"] == 0
}
gnn_caught = {
    r["benchmark"] for r in rows
    if r["true"] == 1 and r["gnn"] == 1
}

print("\nRTL-missed Trojan benchmarks:", sorted(rtl_missed))
print("Trojan benchmarks detected by GNN:", sorted(gnn_caught))
print("GNN catches RTL-missed Trojans:", sorted(rtl_missed & gnn_caught))



RTL only
Accuracy: 0.8846
Balanced accuracy: 0.8846
Macro-F1: 0.8831
Confusion matrix [[TN, FP], [FN, TP]]:
[[26  0]
 [ 6 20]]

GNN only
Accuracy: 0.6538
Balanced accuracy: 0.6538
Macro-F1: 0.6067
Confusion matrix [[TN, FP], [FN, TP]]:
[[26  0]
 [18  8]]

OR fusion (Trojan if either predicts Trojan)
Accuracy: 0.8846
Balanced accuracy: 0.8846
Macro-F1: 0.8831
Confusion matrix [[TN, FP], [FN, TP]]:
[[26  0]
 [ 6 20]]

AND fusion (Trojan if both predict Trojan)
Accuracy: 0.6538
Balanced accuracy: 0.6538
Macro-F1: 0.6067
Confusion matrix [[TN, FP], [FN, TP]]:
[[26  0]
 [18  8]]

RTL-missed Trojan benchmarks: ['AES-T1300', 'AES-T1400', 'AES-T1500', 'AES-T2300', 'AES-T2400', 'AES-T300']
Trojan benchmarks detected by GNN: ['AES-T1100', 'AES-T1200', 'AES-T1600', 'AES-T1700', 'AES-T400', 'AES-T700', 'AES-T800', 'AES-T900']
GNN catches RTL-missed Trojans: []


In [ ]:
import json
from pathlib import Path

for filename in [
    "/content/rtl_only_group_cv_predictions.json",
    "/content/gnn_only_group_cv_predictions.json",
]:
    path = Path(filename)
    rows = json.loads(path.read_text(encoding="utf-8"))

    print("\nFILE:", path.name)
    print("Available fields:", sorted(rows[0].keys()))
    print("Example record:")
    print(json.dumps(rows[0], indent=2))


FILE: rtl_only_group_cv_predictions.json
Available fields: ['benchmark', 'fold', 'predicted_label', 'true_label']
Example record:
{
  "fold": 1,
  "benchmark": "AES-T100",
  "true_label": 0,
  "predicted_label": 0
}

FILE: gnn_only_group_cv_predictions.json
Available fields: ['benchmark', 'fold', 'predicted_label', 'true_label']
Example record:
{
  "fold": 1,
  "benchmark": "AES-T100",
  "true_label": 0,
  "predicted_label": 0
}


In [ ]:
import json
from pathlib import Path
import zipfile

manifest_path = Path("/content/paired_aes_manifest_v1.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print("Paired samples:", len(manifest))
print("Manifest type:", type(manifest).__name__)

if isinstance(manifest, dict):
    print("Top-level keys:", list(manifest.keys()))
    samples = manifest.get("samples", manifest.get("designs", []))
else:
    samples = manifest

print("\nFirst paired sample:")
print(json.dumps(samples[0], indent=2)[:5000])

zip_path = Path("/content/AES_graph_artifacts.zip")
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    json_files = [n for n in names if n.lower().endswith(".json")]
    text_files = [
        n for n in names
        if n.lower().endswith((".v", ".sv", ".vh", ".svh", ".txt", ".csv"))
    ]

    print("\nJSON files inside graph archive:")
    print(json_files[:30])

    print("\nRTL/text files inside graph archive:", len(text_files))
    print(text_files[:30])

    print("\nGraph-related metadata keys:")
    graph_files = [n for n in names if n.endswith(".pt")]
    if graph_files:
        print("Example graph tensor:", graph_files[0])


Paired samples: 7
Manifest type: dict
Top-level keys: ['description', 'excluded_benchmark', 'num_samples', 'label_counts', 'num_unique_rtl_source_hashes', 'duplicate_source_groups', 'samples']

First paired sample:
{
  "benchmark": "AES-T100",
  "label": 0,
  "class_name": "clean",
  "rtl_embedding_path": "clean/AES-T100.pt",
  "rtl_embedding_shape": [
    2,
    768
  ],
  "source_text_sha256": "d839f39773fd8ca68f99771f6fc3de3522466de8d5c99e39f81aa4add575cead",
  "included_files": [
    "clean/AES-T100/aes_128.v",
    "clean/AES-T100/round.v",
    "clean/AES-T100/table.v"
  ],
  "included_file_sha256": {
    "clean/AES-T100/aes_128.v": "6507f2c74cb288e94ae51b2034b2669261893b39067bef3189b2f1a138a8ce52",
    "clean/AES-T100/round.v": "a9c7504563285ded97dc072a653547938b457ea1918b006651d168c61a4ef362",
    "clean/AES-T100/table.v": "36171c0297f95a036285d418b9be3a0d4c4d4e5248a8f0a6133ed7956b37b157"
  },
  "graph_archive_path": "AES/AES-T100_clean_netlist_clean.pt",
  "graph_num_nodes": 790

In [ ]:
import io
import json
import zipfile
import torch
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

BASE = Path("/content")
manifest = json.loads(
    (BASE / "paired_aes_manifest_v1.json").read_text()
)

# Load one clean and one Trojan graph safely
safe_classes = [Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]

with zipfile.ZipFile(BASE / "AES_graph_artifacts.zip") as z:
    graph_names = z.namelist()

    for sample in [
        next(s for s in manifest["samples"] if s["label"] == 0),
        next(s for s in manifest["samples"] if s["label"] == 1),
    ]:
        graph_path = sample["graph_archive_path"]
        matches = [
            n for n in graph_names
            if n.replace("\\", "/").endswith(
                graph_path.replace("\\", "/")
            )
        ]
        assert len(matches) == 1, (
            sample["benchmark"], graph_path, matches
        )

        with z.open(matches[0]) as f:
            raw = io.BytesIO(f.read())

        with torch.serialization.safe_globals(safe_classes):
            graph = torch.load(
                raw, map_location="cpu", weights_only=True
            )

        embedding_path = (
            BASE / "rtl_embeddings_v2" / sample["rtl_embedding_path"]
        )
        rtl = torch.load(
            embedding_path, map_location="cpu", weights_only=True
        )

        print(f"\nBenchmark: {sample['benchmark']}")
        print("Label:", sample["label"])
        print("Node features:", tuple(graph.x.shape))
        print("Edges:", tuple(graph.edge_index.shape))
        print("RTL embeddings:", tuple(rtl.shape))


Benchmark: AES-T100
Label: 0
Node features: (7903, 20)
Edges: (2, 6519)
RTL embeddings: (2, 768)

Benchmark: AES-T100
Label: 1
Node features: (7716, 20)
Edges: (2, 6530)
RTL embeddings: (2, 768)


In [ ]:
import json
from pathlib import Path

path = Path("/content/paired_aes_group_folds_v1.json")
folds = json.loads(path.read_text(encoding="utf-8"))

print("Top-level type:", type(folds).__name__)

if isinstance(folds, dict):
    print("Top-level keys:", list(folds.keys()))
    print(json.dumps(folds, indent=2)[:5000])
else:
    print("Number of entries:", len(folds))
    print(json.dumps(folds[:2], indent=2)[:5000])

Top-level type: dict
Top-level keys: ['method', 'seed', 'note', 'folds']
{
  "method": "two-fold clean-source-group-held-out evaluation",
  "seed": null,
  "note": "Small dataset; only two distinct clean RTL source groups. Report each fold separately and pooled out-of-fold results. Do not use test-fold metrics for model selection.",
  "folds": [
    {
      "fold": 1,
      "train_benchmarks": [
        "AES-T2300",
        "AES-T2400",
        "AES-T2500",
        "AES-T2600",
        "AES-T2700",
        "AES-T2800"
      ],
      "test_benchmarks": [
        "AES-T100",
        "AES-T1000",
        "AES-T1100",
        "AES-T1200",
        "AES-T1300",
        "AES-T1400",
        "AES-T1500",
        "AES-T1600",
        "AES-T1700",
        "AES-T1800",
        "AES-T1900",
        "AES-T200",
        "AES-T2000",
        "AES-T2100",
        "AES-T300",
        "AES-T400",
        "AES-T500",
        "AES-T700",
        "AES-T800",
        "AES-T900"
      ],
      "train_samples

In [ ]:

import io
import json
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

# --------------------------------------------------
# 1. Paths and reproducibility
# --------------------------------------------------
BASE = Path("/content")
MANIFEST_PATH = BASE / "paired_aes_manifest_v1.json"
FOLD_PATH = BASE / "paired_aes_group_folds_v1.json"
GRAPH_ZIP = BASE / "AES_graph_artifacts.zip"
EMBEDDING_ROOT = BASE / "rtl_embeddings_v2"

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    paired_manifest = json.load(f)

with open(FOLD_PATH, "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

samples = paired_manifest["samples"]
assert len(samples) == 52
assert paired_manifest["label_counts"] == {"0": 26, "1": 26}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print("Device:", device)
print("Samples:", len(samples))
print("Fold count:", len(split_manifest["folds"]))


# --------------------------------------------------
# 2. Safe loading of graph and RTL tensors
# --------------------------------------------------
SAFE_CLASSES = [Data, DataEdgeAttr, DataTensorAttr, GlobalStorage]

def load_sample(sample):
    # Load graph from ZIP; normalize Windows/Linux separators.
    wanted = sample["graph_archive_path"].replace("\\", "/")

    with zipfile.ZipFile(GRAPH_ZIP, "r") as z:
        matches = [
            name for name in z.namelist()
            if name.replace("\\", "/").endswith(wanted)
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"Expected one graph for {wanted}; found {len(matches)}"
            )

        with z.open(matches[0]) as f:
            graph_bytes = io.BytesIO(f.read())

        with torch.serialization.safe_globals(SAFE_CLASSES):
            graph = torch.load(
                graph_bytes,
                map_location="cpu",
                weights_only=True
            )

    # Load the precomputed CodeBERT chunk embeddings.
    rtl_path = EMBEDDING_ROOT / sample["rtl_embedding_path"]
    rtl = torch.load(
        rtl_path,
        map_location="cpu",
        weights_only=True
    )

    graph.x = graph.x.float()
    graph.edge_index = graph.edge_index.long()
    rtl = rtl.float()

    assert graph.x.ndim == 2 and graph.x.shape[1] == 20
    assert graph.edge_index.shape[0] == 2
    assert rtl.shape == (2, 768)
    assert torch.isfinite(graph.x).all()
    assert torch.isfinite(rtl).all()

    return graph, rtl


# --------------------------------------------------
# 3. Cross-attention fusion model
# --------------------------------------------------
class CrossAttentionFusion(nn.Module):
    def __init__(self, node_dim=20, rtl_dim=768,
                 hidden_dim=32, num_heads=4, dropout=0.2):
        super().__init__()

        # Graph encoder: preserve node-level representations.
        self.gcn1 = GCNConv(node_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.graph_norm = nn.LayerNorm(hidden_dim)

        # Project CodeBERT embeddings into the shared space.
        self.rtl_projection = nn.Sequential(
            nn.Linear(rtl_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim)
        )

        # Graph nodes query RTL chunks.
        self.graph_queries_rtl = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=dropout,
            batch_first=True
        )

        # RTL chunks query graph nodes.
        self.rtl_queries_graph = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=dropout,
            batch_first=True
        )

        self.graph_cross_norm = nn.LayerNorm(hidden_dim)
        self.rtl_cross_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Graph mean + max + RTL mean + RTL max = 4 * hidden_dim.
        self.classifier = nn.Sequential(
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, graph, rtl):
        # graph.x: [number_of_nodes, 20]
        # rtl:     [number_of_chunks, 768]

        x = F.relu(self.gcn1(graph.x, graph.edge_index))
        x = self.dropout(x)
        node_repr = F.relu(self.gcn2(x, graph.edge_index))
        node_repr = self.graph_norm(node_repr)

        # Add batch dimension for attention: [1, N, H].
        graph_tokens = node_repr.unsqueeze(0)
        rtl_tokens = self.rtl_projection(rtl).unsqueeze(0)

        # Graph-to-RTL attention.
        graph_context, graph_to_rtl_weights = self.graph_queries_rtl(
            query=graph_tokens,
            key=rtl_tokens,
            value=rtl_tokens,
            need_weights=True,
            average_attn_weights=True
        )
        graph_context = self.graph_cross_norm(
            graph_tokens + self.dropout(graph_context)
        )

        # RTL-to-graph attention.
        rtl_context, rtl_to_graph_weights = self.rtl_queries_graph(
            query=rtl_tokens,
            key=graph_tokens,
            value=graph_tokens,
            need_weights=True,
            average_attn_weights=True
        )
        rtl_context = self.rtl_cross_norm(
            rtl_tokens + self.dropout(rtl_context)
        )

        # Global graph summaries; node-level outputs remain available.
        graph_mean = graph_context.mean(dim=1)
        graph_max = graph_context.max(dim=1).values

        rtl_mean = rtl_context.mean(dim=1)
        rtl_max = rtl_context.max(dim=1).values

        fused = torch.cat(
            [graph_mean, graph_max, rtl_mean, rtl_max],
            dim=-1
        )

        logits = self.classifier(fused)

        return {
            "logits": logits,
            "node_embeddings": node_repr,
            "contextual_node_embeddings": graph_context.squeeze(0),
            "rtl_context_embeddings": rtl_context.squeeze(0),
            "graph_to_rtl_attention": graph_to_rtl_weights.squeeze(0),
            "rtl_to_graph_attention": rtl_to_graph_weights.squeeze(0),
            "fused_embedding": fused
        }


# --------------------------------------------------
# 4. Smoke test: one clean and one Trojan graph
# --------------------------------------------------
model = CrossAttentionFusion().to(device)
model.eval()

for label in (0, 1):
    sample = next(s for s in samples if s["label"] == label)
    graph, rtl = load_sample(sample)

    graph = graph.to(device)
    rtl = rtl.to(device)

    with torch.no_grad():
        output = model(graph, rtl)

    print(f"\nBenchmark: {sample['benchmark']}")
    print("True label:", label)
    print("Graph nodes:", graph.x.shape[0])
    print("RTL chunks:", rtl.shape[0])
    print("Logits:", tuple(output["logits"].shape))
    print("Node embeddings:",
          tuple(output["node_embeddings"].shape))
    print("Contextual node embeddings:",
          tuple(output["contextual_node_embeddings"].shape))
    print("Graph-to-RTL attention:",
          tuple(output["graph_to_rtl_attention"].shape))
    print("RTL-to-graph attention:",
          tuple(output["rtl_to_graph_attention"].shape))

print("\nSMOKE TEST PASSED")


Device: cuda
Samples: 52
Fold count: 2

Benchmark: AES-T100
True label: 0
Graph nodes: 7903
RTL chunks: 2
Logits: (1, 2)
Node embeddings: (7903, 32)
Contextual node embeddings: (7903, 32)
Graph-to-RTL attention: (7903, 2)
RTL-to-graph attention: (2, 7903)

Benchmark: AES-T100
True label: 1
Graph nodes: 7716
RTL chunks: 2
Logits: (1, 2)
Node embeddings: (7716, 32)
Contextual node embeddings: (7716, 32)
Graph-to-RTL attention: (7716, 2)
RTL-to-graph attention: (2, 7716)

SMOKE TEST PASSED


In [ ]:

import copy
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

OUT = Path("/content/fusion_group_cv")
OUT.mkdir(parents=True, exist_ok=True)

EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-3

all_predictions = []
fold_summaries = []

for fold_info in split_manifest["folds"]:
    fold_id = fold_info["fold"]
    train_benchmarks = set(fold_info["train_benchmarks"])
    test_benchmarks = set(fold_info["test_benchmarks"])

    train_samples = [
        s for s in samples if s["benchmark"] in train_benchmarks
    ]
    test_samples = [
        s for s in samples if s["benchmark"] in test_benchmarks
    ]

    # Verify that all expected paired samples are present.
    assert len(train_samples) == fold_info["train_samples"].__len__()
    assert len(test_samples) == fold_info["test_samples"].__len__()
    assert not (train_benchmarks & test_benchmarks)

    # Load samples once per fold.
    train_data = [(s, *load_sample(s)) for s in train_samples]
    test_data = [(s, *load_sample(s)) for s in test_samples]

    # New model and optimizer for each fold.
    torch.manual_seed(100 + fold_id)
    model = CrossAttentionFusion().to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    print(f"\n{'=' * 60}")
    print(f"FOLD {fold_id}")
    print("Train samples:", len(train_data))
    print("Test samples:", len(test_data))

    # Fixed training schedule. No test-fold-based early stopping.
    model.train()
    for epoch in range(EPOCHS):
        order = torch.randperm(len(train_data)).tolist()
        total_loss = 0.0

        for idx in order:
            sample, graph, rtl = train_data[idx]
            graph = graph.to(device)
            rtl = rtl.to(device)
            label = torch.tensor(
                [sample["label"]], dtype=torch.long, device=device
            )

            optimizer.zero_grad(set_to_none=True)
            output = model(graph, rtl)
            loss = F.cross_entropy(output["logits"], label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        if epoch in (0, 4, 9, 14, 19, 24):
            print(
                f"Epoch {epoch + 1:02d}/{EPOCHS} "
                f"mean_train_loss={total_loss / len(train_data):.4f}"
            )

    # Evaluate once after the fixed schedule.
    model.eval()
    fold_preds = []

    with torch.no_grad():
        for sample, graph, rtl in test_data:
            graph = graph.to(device)
            rtl = rtl.to(device)
            output = model(graph, rtl)

            probs = torch.softmax(output["logits"], dim=-1)[0].cpu().numpy()
            pred = int(np.argmax(probs))

            row = {
                "fold": fold_id,
                "benchmark": sample["benchmark"],
                "true_label": int(sample["label"]),
                "predicted_label": pred,
                "prob_clean": float(probs[0]),
                "prob_trojan": float(probs[1])
            }
            fold_preds.append(row)
            all_predictions.append(row)

    y_true = [r["true_label"] for r in fold_preds]
    y_pred = [r["predicted_label"] for r in fold_preds]

    summary = {
        "fold": fold_id,
        "n_train": len(train_data),
        "n_test": len(test_data),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).tolist()
    }
    fold_summaries.append(summary)

    print("\nFold metrics:")
    print(json.dumps(summary, indent=2))
    print(classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=["clean", "trojan"],
        zero_division=0
    ))

    torch.save(
        model.state_dict(),
        OUT / f"fusion_fold_{fold_id}.pt"
    )

# Save out-of-fold predictions and fold summaries.
(OUT / "fusion_oof_predictions.json").write_text(
    json.dumps(all_predictions, indent=2), encoding="utf-8"
)
(OUT / "fusion_fold_summaries.json").write_text(
    json.dumps(fold_summaries, indent=2), encoding="utf-8"
)

# Pooled out-of-fold results.
y_true = [r["true_label"] for r in all_predictions]
y_pred = [r["predicted_label"] for r in all_predictions]

pooled = {
    "n_samples": len(all_predictions),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_f1": float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist()
}

(OUT / "fusion_pooled_summary.json").write_text(
    json.dumps(pooled, indent=2), encoding="utf-8"
)

print("\nPOOLED OUT-OF-FOLD RESULTS")
print(json.dumps(pooled, indent=2))
print("\nSaved outputs to:", OUT)
print("Note: metrics are exploratory given the small, two-group dataset.")



FOLD 1
Train samples: 12
Test samples: 40
Epoch 01/25 mean_train_loss=0.7422
Epoch 05/25 mean_train_loss=0.7036
Epoch 10/25 mean_train_loss=0.5109
Epoch 15/25 mean_train_loss=0.1608
Epoch 20/25 mean_train_loss=0.0199
Epoch 25/25 mean_train_loss=0.0091

Fold metrics:
{
  "fold": 1,
  "n_train": 12,
  "n_test": 40,
  "accuracy": 0.9,
  "balanced_accuracy": 0.9,
  "macro_f1": 0.898989898989899,
  "confusion_matrix": [
    [
      20,
      0
    ],
    [
      4,
      16
    ]
  ]
}
              precision    recall  f1-score   support

       clean       0.83      1.00      0.91        20
      trojan       1.00      0.80      0.89        20

    accuracy                           0.90        40
   macro avg       0.92      0.90      0.90        40
weighted avg       0.92      0.90      0.90        40


FOLD 2
Train samples: 40
Test samples: 12
Epoch 01/25 mean_train_loss=1.0114
Epoch 05/25 mean_train_loss=0.0097
Epoch 10/25 mean_train_loss=0.0006
Epoch 15/25 mean_train_loss=0.0003
Epo

In [ ]:

import json
import numpy as np
from pathlib import Path

OUT = Path("/content/fusion_group_cv")
LOCAL_OUT = OUT / "localization_artifacts"
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

localization_manifest = []

for fold_info in split_manifest["folds"]:
    fold_id = fold_info["fold"]
    test_benchmarks = set(fold_info["test_benchmarks"])

    checkpoint = OUT / f"fusion_fold_{fold_id}.pt"
    model = CrossAttentionFusion().to(device)
    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device,
            weights_only=True
        )
    )
    model.eval()

    fold_samples = [
        s for s in samples
        if s["benchmark"] in test_benchmarks
    ]

    for sample in fold_samples:
        graph, rtl = load_sample(sample)
        graph = graph.to(device)
        rtl = rtl.to(device)

        with torch.no_grad():
            result = model(graph, rtl)
            probabilities = torch.softmax(
                result["logits"], dim=-1
            )[0].cpu().numpy()

        # Candidate node-level attention score:
        # mean attention assigned to each graph node by RTL chunks.
        # This is a ranking signal, NOT validated localization.
        node_attention = (
            result["rtl_to_graph_attention"]
            .mean(dim=0)
            .cpu()
            .numpy()
        )

        safe_name = sample["benchmark"] + "_" + sample["class_name"]
        output_path = LOCAL_OUT / f"fold{fold_id}_{safe_name}.npz"

        np.savez_compressed(
            output_path,
            node_features=graph.x.cpu().numpy(),
            edge_index=graph.edge_index.cpu().numpy(),
            node_embeddings=result["node_embeddings"].cpu().numpy(),
            contextual_node_embeddings=(
                result["contextual_node_embeddings"].cpu().numpy()
            ),
            graph_to_rtl_attention=(
                result["graph_to_rtl_attention"].cpu().numpy()
            ),
            rtl_to_graph_attention=(
                result["rtl_to_graph_attention"].cpu().numpy()
            ),
            candidate_node_attention=node_attention,
            logits=result["logits"].cpu().numpy(),
            probabilities=probabilities
        )

        localization_manifest.append({
            "fold": fold_id,
            "benchmark": sample["benchmark"],
            "label": int(sample["label"]),
            "class_name": sample["class_name"],
            "graph_archive_path": sample["graph_archive_path"],
            "rtl_embedding_path": sample["rtl_embedding_path"],
            "num_nodes": int(graph.x.shape[0]),
            "num_edges": int(graph.edge_index.shape[1]),
            "artifact": str(output_path),
            "node_index_mapping_verified": False,
            "localization_status": "candidate_scores_only"
        })

(LOCAL_OUT / "manifest.json").write_text(
    json.dumps(localization_manifest, indent=2),
    encoding="utf-8"
)

print("Artifacts saved:", len(localization_manifest))
print("Directory:", LOCAL_OUT)
print("Manifest:", LOCAL_OUT / "manifest.json")
print("Status: node-level outputs exported; localization not yet validated.")


Artifacts saved: 52
Directory: /content/fusion_group_cv/localization_artifacts
Manifest: /content/fusion_group_cv/localization_artifacts/manifest.json
Status: node-level outputs exported; localization not yet validated.


In [ ]:
from pathlib import Path
import zipfile
import json

base = Path("/content/fusion_group_cv")
assert (base / "fusion_fold_1.pt").exists()
assert (base / "fusion_fold_2.pt").exists()
assert (base / "fusion_oof_predictions.json").exists()
assert (base / "fusion_pooled_summary.json").exists()
assert (base / "localization_artifacts/manifest.json").exists()

manifest = json.loads(
    (base / "localization_artifacts/manifest.json").read_text()
)
assert len(manifest) == 52

archive_path = Path("/content/fusion_implementation_handoff.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for path in base.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(base.parent))

print("Handoff archive:", archive_path)
print("Localization artifacts:", len(manifest))
print("Archive size (MB):", round(archive_path.stat().st_size / 1e6, 2))
print("PACKAGE CHECK PASSED")

Handoff archive: /content/fusion_implementation_handoff.zip
Localization artifacts: 52
Archive size (MB): 5.43
PACKAGE CHECK PASSED
